# Semana 3: Operaciones con Tensores - Construyendo Algoritmos
## Notebook de Ejercicios (NB2) - Dataset MNIST

### Propósito de la sesión
Aplicar las operaciones fundamentales con tensores (reducción, producto punto, similitud de coseno) a un dataset real de dígitos escritos a mano. Implementaremos desde cero un clasificador k-NN basado en similitud de coseno y lo compararemos con la implementación de scikit-learn.

### Objetivos de aprendizaje
*   Cargar y explorar el dataset MNIST como tensores.
*   Calcular la **imagen promedio** de cada dígito usando operaciones de reducción.
*   Implementar **similitud de coseno** para encontrar los dígitos más similares a una consulta.
*   Construir un clasificador **k-NN** manual usando producto punto.
*   Comparar nuestra implementación con `sklearn.neighbors.KNeighborsClassifier`.
*   Visualizar resultados y comprender las fortalezas/limitaciones de k-NN.

## Configuración Inicial

Importamos las librerías necesarias y cargamos MNIST. Usaremos `keras.datasets` para facilitar la carga.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import time

# Para cargar MNIST
try:
    from tensorflow.keras.datasets import mnist
    print("✅ Keras importado correctamente.")
except ImportError:
    print("❌ TensorFlow/Keras no encontrado. Instalando...")
    !pip install tensorflow
    from tensorflow.keras.datasets import mnist
    print("✅ TensorFlow/Keras instalado e importado.")

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print("\n🎯 Librerías listas. Cargando dataset MNIST...")

# Cargamos MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Exploramos las dimensiones
print(f"\n🔷 MNIST - Dataset de dígitos escritos a mano")
print(f"  Tensor de entrenamiento (X_train) - forma: {X_train.shape}")
print(f"  Tensor de prueba (X_test) - forma: {X_test.shape}")
print(f"  Etiquetas de entrenamiento (y_train) - forma: {y_train.shape}")
print(f"  Etiquetas de prueba (y_test) - forma: {y_test.shape}")

# Normalizamos los datos (valores entre 0 y 1)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"\n📊 Datos normalizados: rango [{X_train.min():.2f}, {X_train.max():.2f}]")

### Visualización de muestras

Visualicemos algunas imágenes del dataset para familiarizarnos con los datos.

In [ ]:
def mostrar_imagenes(imagenes, etiquetas, num_imagenes=10):
    plt.figure(figsize=(15, 6))
    for i in range(num_imagenes):
        plt.subplot(2, 5, i+1)
        plt.imshow(imagenes[i], cmap='gray')
        plt.title(f"Dígito: {etiquetas[i]}")
        plt.axis('off')
    plt.suptitle("Muestras del Dataset MNIST", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

mostrar_imagenes(X_train, y_train, num_imagenes=10)

---
## Actividad 1: Calcular la imagen promedio de cada dígito (usando reducción)

Vamos a calcular el **promedio** de todas las imágenes para cada dígito (0-9). Esto nos dará una imagen "prototipo" que representa la forma típica de cada dígito.

Usaremos operaciones de **reducción** (media) y enmascaramiento (slicing lógico).

In [ ]:
# Inicializamos un arreglo para almacenar los promedios
digitos = np.unique(y_train)
imagenes_promedio = []

plt.figure(figsize=(15, 6))
for i, digito in enumerate(digitos):
    # Seleccionamos todas las imágenes de este dígito (enmascaramiento)
    mascara = (y_train == digito)
    imagenes_digito = X_train[mascara]  # shape: (num_ejemplares, 28, 28)
    
    # Calculamos el promedio a lo largo del eje de las muestras (axis=0)
    # Esto es una reducción: de (N, 28, 28) a (28, 28)
    promedio = np.mean(imagenes_digito, axis=0)
    imagenes_promedio.append(promedio)
    
    # Mostramos el promedio
    plt.subplot(2, 5, i+1)
    plt.imshow(promedio, cmap='gray')
    plt.title(f'Promedio dígito {digito}\n(n={len(imagenes_digito)})')
    plt.axis('off')

plt.suptitle('Imagen Promedio de Cada Dígito en MNIST', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Convertimos a arreglo numpy para uso posterior
imagenes_promedio = np.array(imagenes_promedio)
print(f"Shape del tensor de promedios: {imagenes_promedio.shape} (10 dígitos, 28x28 cada uno)")

### Análisis de los promedios

Observa cómo los promedios capturan la **esencia** de cada dígito:
*   El **0** tiene un óvalo característico.
*   El **1** es una línea vertical.
*   El **4** muestra su forma puntiaguda.

Estas imágenes promedio pueden verse como **prototipos** o **centroides** de cada clase.

**Pregunta reflexiva:** ¿Por qué el promedio del 7 y del 9 son tan definidos, mientras que el del 8 parece más difuso? (Pista: piensa en la variabilidad de escritura de cada dígito).

---
## Actividad 2: Para un dígito dado, encontrar los 5 dígitos más similares usando similitud de coseno

Ahora usaremos **similitud de coseno** para encontrar, dentro del conjunto de entrenamiento, las imágenes más parecidas a una imagen de consulta.

Recordemos que la similitud de coseno se define como:
$$\text{cosim}(a, b) = \frac{a \cdot b}{\|a\| \|b\|}$$

In [ ]:
# Función para calcular similitud de coseno entre dos vectores
def cos_sim(a, b):
    # Aplanamos las imágenes a vectores 1D
    a_flat = a.flatten()
    b_flat = b.flatten()
    dot = np.dot(a_flat, b_flat)
    norm_a = np.linalg.norm(a_flat)
    norm_b = np.linalg.norm(b_flat)
    return dot / (norm_a * norm_b + 1e-8)  # +epsilon para evitar división por cero

# Elegimos una imagen de consulta (puedes cambiar el índice)
query_idx = 42  # Por ejemplo
query_img = X_test[query_idx]
query_label = y_test[query_idx]

print(f"🔍 Imagen de consulta - Índice {query_idx}, Dígito real: {query_label}")
plt.imshow(query_img, cmap='gray')
plt.title(f'Imagen de Consulta (dígito {query_label})')
plt.axis('off')
plt.show()

# Calculamos la similitud con todas las imágenes de entrenamiento
# (esto puede tomar unos segundos si usamos todo MNIST, así que usaremos un subconjunto)
n_samples = 5000  # Usamos 5000 imágenes para que sea rápido
X_subset = X_train[:n_samples]
y_subset = y_train[:n_samples]

print(f"\nCalculando similitudes con {n_samples} imágenes de entrenamiento...")
similitudes = []
for img in X_subset:
    similitudes.append(cos_sim(query_img, img))

similitudes = np.array(similitudes)

# Encontramos los índices de las 5 imágenes más similares (mayor similitud)
top_5_indices = np.argsort(similitudes)[-5:][::-1]
top_5_sim = similitudes[top_5_indices]

print(f"\n🎯 Las 5 imágenes más similares (similitud de coseno):")
for i, idx in enumerate(top_5_indices):
    print(f"  {i+1}. Índice {idx}, Dígito: {y_subset[idx]}, Similitud: {top_5_sim[i]:.4f}")

# Visualizamos las más similares
plt.figure(figsize=(15, 4))
plt.subplot(1, 6, 1)
plt.imshow(query_img, cmap='gray')
plt.title('Consulta')
plt.axis('off')

for i, idx in enumerate(top_5_indices):
    plt.subplot(1, 6, i+2)
    plt.imshow(X_subset[idx], cmap='gray')
    plt.title(f'#{i+1}: {y_subset[idx]}\nsim={top_5_sim[i]:.3f}')
    plt.axis('off')

plt.suptitle(f'Imagen de Consulta y las 5 Más Similares (dígito real consulta: {query_label})', fontsize=14)
plt.tight_layout()
plt.show()

### Análisis de resultados

Observa si las imágenes más similares corresponden al mismo dígito que la consulta. Si no es así, ¿por qué crees que ocurre? 

*   Algunos dígitos pueden tener formas similares (ej. 3 y 8, 4 y 9).
*   La similitud de coseno mide la orientación (ángulo) más que la magnitud, lo que puede ser útil pero no perfecto.

**Pregunta reflexiva:** ¿Qué similitud tendría la imagen consigo misma? Debería ser 1.0.

---
## Actividad 3: Implementar una predicción "k-NN" usando producto punto y comparar con sklearn

Ahora construiremos un clasificador **k-NN** (k vecinos más cercanos) manualmente, usando **producto punto** (que es equivalente a similitud de coseno si los vectores están normalizados).

**Nota:** Para usar producto punto como medida de similitud, necesitamos que los vectores tengan norma unitaria. Si no, el producto punto favorece vectores con mayor magnitud.

In [ ]:
# Normalizamos las imágenes para que tengan norma L2 = 1 (vectores unitarios)
# Así, el producto punto es exactamente la similitud de coseno.

def normalizar_l2(imagenes):
    # Aplanamos: de (N, 28, 28) a (N, 784)
    imagenes_flat = imagenes.reshape(imagenes.shape[0], -1)
    # Calculamos norma L2 por fila
    normas = np.linalg.norm(imagenes_flat, axis=1, keepdims=True)
    # Normalizamos (evitamos división por cero)
    imagenes_norm = imagenes_flat / (normas + 1e-8)
    return imagenes_norm

# Tomamos subconjuntos para que sea manejable
n_train = 5000
n_test = 500

X_train_norm = normalizar_l2(X_train[:n_train])
y_train_sub = y_train[:n_train]
X_test_norm = normalizar_l2(X_test[:n_test])
y_test_sub = y_test[:n_test]

print(f"Shape X_train_norm: {X_train_norm.shape}")
print(f"Shape X_test_norm: {X_test_norm.shape}")
print(f"Norma L2 de la primera imagen de entrenamiento: {np.linalg.norm(X_train_norm[0]):.4f} (debe ser 1)")

### 3.1. Implementación manual de k-NN

Nuestra implementación:
1. Para cada imagen de prueba, calcular el **producto punto** con todas las imágenes de entrenamiento (esto da la similitud de coseno, ya que están normalizadas).
2. Encontrar los índices de los **k** productos punto más altos (mayor similitud).
3. Obtener las etiquetas de esos k vecinos.
4. Asignar la clase por **voto mayoritario**.

In [ ]:
def knn_manual(X_train, y_train, X_test, k=5):
    """
    Clasificador k-NN manual usando producto punto.
    Asume que X_train y X_test ya están normalizados.
    """
    predictions = []
    
    # Producto punto entre todas las pruebas y todos los entrenamientos
    # X_test: (n_test, d), X_train.T: (d, n_train) -> (n_test, n_train)
    sim_matrix = np.dot(X_test, X_train.T)
    
    for i in range(len(X_test)):
        # Similitudes de la i-ésima prueba con todos los entrenamientos
        sim = sim_matrix[i]
        
        # Índices de los k mayores (argsort da ascendente, tomamos los últimos k)
        top_k_indices = np.argsort(sim)[-k:][::-1]
        
        # Etiquetas de esos k vecinos
        top_k_labels = y_train[top_k_indices]
        
        # Voto mayoritario
        unique, counts = np.unique(top_k_labels, return_counts=True)
        pred = unique[np.argmax(counts)]
        predictions.append(pred)
    
    return np.array(predictions)

# Probamos nuestra implementación
k = 5
print(f"🔷 Ejecutando k-NN manual con k={k}...")
start_time = time.time()
y_pred_manual = knn_manual(X_train_norm, y_train_sub, X_test_norm, k=k)
manual_time = time.time() - start_time

accuracy_manual = accuracy_score(y_test_sub, y_pred_manual)
print(f"✅ k-NN manual completado en {manual_time:.2f} segundos")
print(f"📊 Precisión manual: {accuracy_manual * 100:.2f}%")

### 3.2. Comparación con sklearn

Ahora usamos la implementación optimizada de scikit-learn y comparamos resultados.

In [ ]:
# Usamos KNeighborsClassifier de sklearn
# Nota: por defecto usa métrica minkowski con p=2 (distancia euclidiana).
# Para usar similitud de coseno, configuramos metric='cosine' (internamente sklearn usa 1 - cos_sim)

print(f"🔶 Ejecutando k-NN de sklearn con k={k}...")
knn_sklearn = KNeighborsClassifier(n_neighbors=k, metric='cosine')

start_time = time.time()
knn_sklearn.fit(X_train_norm, y_train_sub)
y_pred_sklearn = knn_sklearn.predict(X_test_norm)
sklearn_time = time.time() - start_time

accuracy_sklearn = accuracy_score(y_test_sub, y_pred_sklearn)
print(f"✅ k-NN sklearn completado en {sklearn_time:.2f} segundos")
print(f"📊 Precisión sklearn: {accuracy_sklearn * 100:.2f}%")

In [ ]:
# Comparación detallada
print("\n📋 COMPARACIÓN DE RESULTADOS")
print(f"{'Método':<20} {'Precisión':<15} {'Tiempo (s)':<15}")
print("-" * 50)
print(f"{'k-NN Manual':<20} {accuracy_manual * 100:<15.2f} {manual_time:<15.2f}")
print(f"{'k-NN sklearn':<20} {accuracy_sklearn * 100:<15.2f} {sklearn_time:<15.2f}")

# Verificamos si las predicciones coinciden
coincidencias = np.mean(y_pred_manual == y_pred_sklearn)
print(f"\n🔍 Las predicciones coinciden en un {coincidencias * 100:.2f}% de los casos.")

if not np.array_equal(y_pred_manual, y_pred_sklearn):
    print("\n⚠️  Hay diferencias. Esto puede deberse a:")
    print("   - Manejo de empates (ties) en el voto mayoritario.")
    print("   - Pequeñas diferencias numéricas en el cálculo de similitud.")
    print("   - sklearn puede usar algoritmos optimizados (KD-Tree, Ball Tree).")

### 3.3. Visualización de errores

Veamos algunos ejemplos donde nuestra implementación manual falló (si los hay) para entender las limitaciones.

In [ ]:
# Encontramos índices donde la predicción manual fue incorrecta
errores_idx = np.where(y_pred_manual != y_test_sub)[0]

print(f"Total de errores en k-NN manual: {len(errores_idx)} de {len(y_test_sub)}")

# Mostramos algunos ejemplos de error
num_errores_mostrar = min(5, len(errores_idx))
if num_errores_mostrar > 0:
    plt.figure(figsize=(15, 4 * ((num_errores_mostrar + 1) // 2)))
    for i, idx in enumerate(errores_idx[:num_errores_mostrar]):
        plt.subplot(2, 3, i+1)
        plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
        plt.title(f'Real: {y_test_sub[idx]}, Pred: {y_pred_manual[idx]}')
        plt.axis('off')
    plt.suptitle('Ejemplos de Error en k-NN Manual', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("🎉 ¡Sin errores en nuestra implementación manual!")

---
## Ejercicios Adicionales para el Estudiante

1.  **Variar k:** Experimenta con diferentes valores de k (1, 3, 5, 7, 10, 20) en ambas implementaciones (manual y sklearn). ¿Cómo afecta k a la precisión? ¿Qué k da el mejor resultado para este subconjunto?

2.  **Usar distancia Euclidiana:** Modifica la implementación manual para usar **distancia euclidiana** (L2) en lugar de similitud de coseno. Recuerda que ahora quieres los vecinos **más cercanos** (menor distancia). Compara resultados.

3.  **Análisis de tiempo:** Ejecuta la implementación manual con diferentes tamaños de conjunto de entrenamiento (1000, 5000, 10000). ¿Cómo escala el tiempo de ejecución? ¿Por qué sklearn es más rápido?

4.  **Prototipos vs Vecinos:** Clasifica una imagen de prueba usando solo el **prototipo** (imagen promedio) de cada clase calculado en la Actividad 1. ¿Qué precisión obtienes? Compara con k-NN.

5.  **Reflexión:** ¿En qué situaciones crees que k-NN basado en similitud de coseno funcionaría mal? (Pista: piensa en datasets con alta dimensionalidad y la maldición de la dimensionalidad).

---
## Conclusiones de la Sesión

*   Hemos aplicado **operaciones de reducción** (media) para calcular las **imágenes promedio** de cada dígito en MNIST, obteniendo prototipos visuales de cada clase.
*   Implementamos la **similitud de coseno** para encontrar las imágenes más similares a una consulta, y visualizamos los resultados, comprendiendo su interpretación geométrica.
*   Construimos un **clasificador k-NN desde cero** usando producto punto (equivalente a coseno con vectores normalizados), pasando por todos los pasos: cálculo de similitudes, selección de vecinos y voto mayoritario.
*   Comparamos nuestra implementación con la de **scikit-learn**, observando que sklearn es más rápido y robusto, pero nuestra versión manual nos permitió entender el algoritmo en profundidad.
*   Este ejercicio integra todos los conceptos de la semana (reducción, producto punto, similitud de coseno) en un pipeline completo de machine learning.

¡Has construido tu primer clasificador desde cero usando solo operaciones fundamentales con tensores!